# Bulk vs NucSeq Age-Correlation Gene Overlap Analysis

Bulk 데이터(Pearson)와 NucSeq 데이터(Pearson / Spearman, cell type별)에서
유의미한 age-correlated 유전자의 겹침을 비교합니다.

**데이터 경로**: `Normal_vs_MASLD_corr_results/`

**분석 구조**
- Bulk: Normal / MASLD 조건 (Pearson 상관계수, BH adjusted p-value)
- NucSeq: Normal / MASLD 조건 × Pearson / Spearman × Cell type 별
- 비교: Bulk sig 유전자 ∩ NucSeq sig 유전자 (Venn diagram + 상세 테이블)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_venn as venn_lib
from matplotlib_venn import venn2
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── 경로 설정 ──────────────────────────────────────────────────────────
DATA_DIR = Path('/home/jaehyunchoi/Liver_bio_project/04_Nucseq_analysis/03_script/Normal_vs_MASLD_corr_results')
OUT_DIR  = DATA_DIR / 'overlap_analysis'
OUT_DIR.mkdir(exist_ok=True)

# ── 유의성 기준 ────────────────────────────────────────────────────────
PVAL_CUTOFF = 0.1   # BH adjusted p-value
CORR_CUTOFF = 0.0    # 절대값 상관계수 최소값 (0 = 필터 없음)

print(f'Data dir : {DATA_DIR}')
print(f'Output   : {OUT_DIR}')
print(f'p-value cutoff : {PVAL_CUTOFF}  |  |corr| cutoff : {CORR_CUTOFF}')

## 1. Bulk 데이터 로드

In [ ]:
# ── Bulk 파일 읽기 (Spearman_bulk 시트) ───────────────────────────────
_raw = pd.read_excel(DATA_DIR / 'MASLD_Bulk_age_correlation.xlsx',
                     sheet_name='Spearman_bulk', header=None)

# row1(condition)의 NaN을 forward-fill → 'Normal', 'Normal', ..., 'MASLD', 'MASLD', ...
cond_row = _raw.iloc[1].ffill()

# 컬럼명 재구성: row1=condition(ffilled), row2=metric
_cols = []
for c in range(_raw.shape[1]):
    cond   = str(cond_row.iloc[c]).strip()
    metric = str(_raw.iloc[2, c]).strip()
    if metric in ('ens_id', 'symbol'):
        _cols.append(metric)
    else:
        _cols.append(f'{cond}_{metric}')

bulk_df = _raw.iloc[3:].copy()
bulk_df.columns = _cols
bulk_df = bulk_df.reset_index(drop=True)

# 숫자 컬럼 형변환 (컬럼별로 처리)
for col in bulk_df.columns:
    if col not in ('ens_id', 'symbol'):
        bulk_df[col] = pd.to_numeric(bulk_df[col], errors='coerce')

print('Bulk shape:', bulk_df.shape)
print(bulk_df.columns.tolist())
bulk_df.head()

In [ ]:
# ── 조건별 유의 유전자 추출 함수 ──────────────────────────────────────
def get_bulk_sig(condition: str,
                 pval_cutoff: float = PVAL_CUTOFF,
                 corr_cutoff: float = CORR_CUTOFF) -> set:
    """Bulk에서 condition(Normal/MASLD) 유의 유전자 symbol 집합 반환."""
    bh_col   = f'{condition}_BH'
    corr_col = f'{condition}_spearman_r'
    mask = (
        bulk_df[bh_col].notna() &
        (bulk_df[bh_col] < pval_cutoff) &
        (bulk_df[corr_col].abs() >= corr_cutoff)
    )
    return set(bulk_df.loc[mask, 'symbol'].dropna())

bulk_normal_sig = get_bulk_sig('Normal')
bulk_masld_sig  = get_bulk_sig('MASLD')

print(f'Bulk Normal sig genes : {len(bulk_normal_sig)}')
print(f'Bulk MASLD  sig genes : {len(bulk_masld_sig)}')

## 2. NucSeq 데이터 로드

In [ ]:
# ── cell type 목록 / 방법 ─────────────────────────────────────────────
METHODS   = ['Pearson', 'Spearman']
CELLTYPES = ['All', 'B_cells', 'Cholangiocyte', 'Endothelial',
             'Hepatocyte', 'Lymphocyte', 'Macrophage', 'Stellete']

# ── NucSeq 전체 로드 ──────────────────────────────────────────────────
nucseq: dict[tuple, pd.DataFrame] = {}   # key: (method, celltype)

for method in METHODS:
    for ct in CELLTYPES:
        fname = DATA_DIR / f'{method}_{ct}_gene_sets.xlsx'
        if fname.exists():
            df = pd.read_excel(fname)
            num_c = [c for c in df.columns if c != 'gene']
            df[num_c] = df[num_c].apply(pd.to_numeric, errors='coerce')
            nucseq[(method, ct)] = df
        else:
            print(f'  [WARN] Not found: {fname.name}')

print(f'Loaded {len(nucseq)} NucSeq files')
# 첫 번째 파일 미리보기
list(nucseq.values())[0].head()

In [ ]:
# ── NucSeq 유의 유전자 추출 함수 ─────────────────────────────────────
def get_nucseq_sig(method: str, celltype: str,
                   condition: str,
                   pval_cutoff: float = PVAL_CUTOFF,
                   corr_cutoff: float = CORR_CUTOFF) -> set:
    """NucSeq에서 method/celltype/condition 유의 유전자 집합 반환."""
    df = nucseq.get((method, celltype))
    if df is None:
        return set()
    pval_col = f'adj_pval_{condition.lower()}'
    corr_col = f'{condition.lower()}_corr'
    mask = (
        df[pval_col].notna() &
        (df[pval_col] < pval_cutoff) &
        (df[corr_col].abs() >= corr_cutoff)
    )
    return set(df.loc[mask, 'gene'].dropna())

# 샘플 확인
ex = get_nucseq_sig('Pearson', 'Hepatocyte', 'Normal')
print(f'Pearson/Hepatocyte/Normal sig: {len(ex)}')

## 3. 유의 유전자 수 요약 테이블

In [ ]:
rows = []
for method in METHODS:
    for ct in CELLTYPES:
        for cond in ['Normal', 'MASLD']:
            ns = get_nucseq_sig(method, ct, cond)
            rows.append({'Method': method, 'CellType': ct, 'Condition': cond,
                         'NucSeq_sig': len(ns)})

summary_df = pd.DataFrame(rows)

# Pivot
pivot = summary_df.pivot_table(
    index='CellType', columns=['Method', 'Condition'], values='NucSeq_sig'
)
print('\n[NucSeq significant gene counts]')
display(pivot)

print(f'\nBulk Normal sig: {len(bulk_normal_sig)}')
print(f'Bulk MASLD  sig: {len(bulk_masld_sig)}')

## 4. 겹치는 유전자 계산 (Bulk ∩ NucSeq)

In [ ]:
overlap_records = []

for method in METHODS:
    for ct in CELLTYPES:
        for cond in ['Normal', 'MASLD']:
            bulk_sig  = bulk_normal_sig if cond == 'Normal' else bulk_masld_sig
            nucseq_sig = get_nucseq_sig(method, ct, cond)
            overlap   = bulk_sig & nucseq_sig
            overlap_records.append({
                'Method'       : method,
                'CellType'     : ct,
                'Condition'    : cond,
                'Bulk_sig'     : len(bulk_sig),
                'NucSeq_sig'   : len(nucseq_sig),
                'Overlap'      : len(overlap),
                'Overlap_genes': sorted(overlap),
                'Overlap_pct_bulk'  : round(len(overlap) / len(bulk_sig)  * 100, 2) if bulk_sig  else 0,
                'Overlap_pct_nucseq': round(len(overlap) / len(nucseq_sig) * 100, 2) if nucseq_sig else 0,
            })

overlap_df = pd.DataFrame(overlap_records)
print('Overlap table shape:', overlap_df.shape)
overlap_df.drop(columns='Overlap_genes').head(10)

In [ ]:
# ── 겹침 수 Heatmap (Pearson / Spearman 나란히) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, method in zip(axes, METHODS):
    sub = overlap_df[overlap_df['Method'] == method].copy()
    heat = sub.pivot_table(index='CellType', columns='Condition', values='Overlap', aggfunc='sum')
    sns.heatmap(heat, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
                linewidths=0.5, cbar_kws={'label': '# Overlap genes'})
    ax.set_title(f'{method}  –  Bulk ∩ NucSeq overlap (p < {PVAL_CUTOFF})',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Cell Type')

plt.suptitle('Number of significant overlapping genes\n(Bulk Pearson vs NucSeq by method & cell type)',
             y=1.03, fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / 'overlap_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: overlap_heatmap.png')

In [ ]:
# ── Overlap % (of Bulk) Heatmap ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

for ax, method in zip(axes, METHODS):
    sub = overlap_df[overlap_df['Method'] == method].copy()
    heat = sub.pivot_table(index='CellType', columns='Condition',
                            values='Overlap_pct_bulk', aggfunc='mean')
    sns.heatmap(heat, annot=True, fmt='.1f', cmap='Blues', ax=ax,
                linewidths=0.5, vmin=0, vmax=100,
                cbar_kws={'label': '% of Bulk sig genes recovered'})
    ax.set_title(f'{method}  –  Overlap / Bulk sig (%)', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Cell Type')

plt.suptitle('Overlap as % of Bulk significant genes', y=1.03, fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / 'overlap_pct_bulk_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: overlap_pct_bulk_heatmap.png')

## 5. Venn Diagram (cell type × condition × method)

In [ ]:
for cond in ['Normal', 'MASLD']:
    bulk_sig = bulk_normal_sig if cond == 'Normal' else bulk_masld_sig

    for method in METHODS:
        n_ct   = len(CELLTYPES)
        ncols  = 4
        nrows  = (n_ct + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(ncols * 4, nrows * 4))
        axes = axes.flatten()

        for i, ct in enumerate(CELLTYPES):
            ax = axes[i]
            nucseq_sig = get_nucseq_sig(method, ct, cond)
            overlap    = bulk_sig & nucseq_sig

            v = venn2(
                subsets=(len(bulk_sig - nucseq_sig),
                         len(nucseq_sig - bulk_sig),
                         len(overlap)),
                set_labels=(f'Bulk\n({len(bulk_sig)})',
                             f'NucSeq\n({len(nucseq_sig)})'),
                ax=ax
            )
            ax.set_title(ct, fontsize=11, fontweight='bold')

        # 빈 subplot 숨기기
        for j in range(n_ct, len(axes)):
            axes[j].set_visible(False)

        fig.suptitle(f'Bulk ∩ NucSeq ({method})  –  {cond}  |  p < {PVAL_CUTOFF}',
                     fontsize=14, fontweight='bold', y=1.01)
        plt.tight_layout()
        fname = OUT_DIR / f'venn_{method}_{cond}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fname.name}')

## 6. Pearson vs Spearman 비교 (같은 cell type, 같은 condition)

In [ ]:
# ── Pearson overlap ∩ Spearman overlap (즉, 세 데이터셋 모두에서 겹치는 유전자) ──
method_cmp_rows = []

for ct in CELLTYPES:
    for cond in ['Normal', 'MASLD']:
        bulk_sig = bulk_normal_sig if cond == 'Normal' else bulk_masld_sig

        ov_p = bulk_sig & get_nucseq_sig('Pearson',  ct, cond)
        ov_s = bulk_sig & get_nucseq_sig('Spearman', ct, cond)

        method_cmp_rows.append({
            'CellType'           : ct,
            'Condition'          : cond,
            'Overlap_Pearson'    : len(ov_p),
            'Overlap_Spearman'   : len(ov_s),
            'Overlap_Both'       : len(ov_p & ov_s),
            'Only_Pearson'       : len(ov_p - ov_s),
            'Only_Spearman'      : len(ov_s - ov_p),
            'Overlap_Both_genes' : sorted(ov_p & ov_s),
            'Only_Pearson_genes' : sorted(ov_p - ov_s),
            'Only_Spearman_genes': sorted(ov_s - ov_p),
        })

mcmp_df = pd.DataFrame(method_cmp_rows)
mcmp_df.drop(columns=[c for c in mcmp_df.columns if c.endswith('_genes')]).head(10)

In [ ]:
# ── Pearson vs Spearman overlap 비교 bar chart ────────────────────────
for cond in ['Normal', 'MASLD']:
    sub = mcmp_df[mcmp_df['Condition'] == cond].set_index('CellType')
    plot_cols = ['Only_Pearson', 'Overlap_Both', 'Only_Spearman']
    colors    = ['#4C72B0', '#55A868', '#C44E52']

    fig, ax = plt.subplots(figsize=(12, 5))
    sub[plot_cols].plot(kind='bar', stacked=True, ax=ax,
                        color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{cond}  –  Pearson vs Spearman overlap with Bulk\n'
                 f'(p < {PVAL_CUTOFF})', fontsize=13, fontweight='bold')
    ax.set_xlabel('Cell Type')
    ax.set_ylabel('# Genes')
    ax.legend(['Only in Pearson overlap', 'Both Pearson & Spearman', 'Only in Spearman overlap'],
              loc='upper right')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    fname = OUT_DIR / f'pearson_vs_spearman_overlap_{cond}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname.name}')

## 7. 상세 유전자 테이블 저장

In [ ]:
# ── Bulk+NucSeq 정보 합쳐서 상세 테이블 생성 ─────────────────────────
detail_rows = []

for _, row in overlap_df.iterrows():
    method = row['Method']
    ct     = row['CellType']
    cond   = row['Condition']
    genes  = row['Overlap_genes']

    if not genes:
        continue

    # Bulk 정보
    bulk_sub = bulk_df.set_index('symbol')
    # NucSeq 정보
    ns_df = nucseq.get((method, ct))
    if ns_df is not None:
        ns_sub = ns_df.set_index('gene')

    for gene in genes:
        r = {'Method': method, 'CellType': ct, 'Condition': cond, 'Gene': gene}

        # Bulk 컬럼
        if gene in bulk_sub.index:
            r['Bulk_corr']   = bulk_sub.at[gene, f'{cond}_pearson_r']
            r['Bulk_BH']     = bulk_sub.at[gene, f'{cond}_BH']

        # NucSeq 컬럼
        cond_l = cond.lower()
        if ns_df is not None and gene in ns_sub.index:
            r['NucSeq_corr'] = ns_sub.at[gene, f'{cond_l}_corr']
            r['NucSeq_adjp'] = ns_sub.at[gene, f'adj_pval_{cond_l}']

        detail_rows.append(r)

detail_df = pd.DataFrame(detail_rows)
print('Detail table shape:', detail_df.shape)
detail_df.head(10)

In [ ]:
# ── Excel 저장: sheet = Method_Condition ─────────────────────────────
out_xlsx = OUT_DIR / 'Bulk_NucSeq_overlap_genes.xlsx'

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    # 요약 시트
    overlap_df.drop(columns='Overlap_genes').to_excel(
        writer, sheet_name='Summary', index=False)
    
    # method×condition 별 상세 시트
    for method in METHODS:
        for cond in ['Normal', 'MASLD']:
            sub = detail_df[
                (detail_df['Method'] == method) &
                (detail_df['Condition'] == cond)
            ].sort_values(['CellType', 'NucSeq_adjp'])
            sheet = f'{method}_{cond}'
            sub.to_excel(writer, sheet_name=sheet, index=False)

    # Pearson vs Spearman 비교 시트
    mcmp_df.drop(columns=[c for c in mcmp_df.columns if c.endswith('_genes')]).to_excel(
        writer, sheet_name='PearsonVsSpearman_summary', index=False)

    # Pearson vs Spearman – 양쪽 모두에서 겹치는 유전자 목록
    both_rows = []
    for _, r in mcmp_df.iterrows():
        for g in r['Overlap_Both_genes']:
            both_rows.append({'CellType': r['CellType'],
                               'Condition': r['Condition'], 'Gene': g,
                               'Note': 'Bulk+Pearson+Spearman'})
    pd.DataFrame(both_rows).to_excel(
        writer, sheet_name='Both_methods_overlap', index=False)

print(f'Saved: {out_xlsx}')

## 8. Pearson & Spearman 둘 다에서 Bulk와 겹치는 유전자 (세 방법 공통)

In [ ]:
print('=== Bulk + Pearson NucSeq + Spearman NucSeq 모두에서 유의한 유전자 ===\n')

for cond in ['Normal', 'MASLD']:
    print(f'── {cond} ──')
    bulk_sig = bulk_normal_sig if cond == 'Normal' else bulk_masld_sig

    for ct in CELLTYPES:
        ov_p = bulk_sig & get_nucseq_sig('Pearson',  ct, cond)
        ov_s = bulk_sig & get_nucseq_sig('Spearman', ct, cond)
        both = ov_p & ov_s
        if both:
            print(f'  {ct:15s}: {len(both):4d} genes  →  {sorted(both)[:10]}{"..." if len(both)>10 else ""}')
        else:
            print(f'  {ct:15s}: 0 genes')
    print()

In [ ]:
# ── 3-set Venn (Bulk / Pearson NucSeq / Spearman NucSeq) per cell type ──
try:
    from matplotlib_venn import venn3

    for cond in ['Normal', 'MASLD']:
        bulk_sig = bulk_normal_sig if cond == 'Normal' else bulk_masld_sig
        n_ct   = len(CELLTYPES)
        ncols  = 4
        nrows  = (n_ct + ncols - 1) // ncols

        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(ncols * 4.5, nrows * 4.5))
        axes = axes.flatten()

        for i, ct in enumerate(CELLTYPES):
            ax  = axes[i]
            p_s = get_nucseq_sig('Pearson',  ct, cond)
            s_s = get_nucseq_sig('Spearman', ct, cond)
            venn3(
                subsets=(bulk_sig, p_s, s_s),
                set_labels=('Bulk', 'NucSeq\nPearson', 'NucSeq\nSpearman'),
                ax=ax
            )
            ax.set_title(ct, fontsize=11, fontweight='bold')

        for j in range(n_ct, len(axes)):
            axes[j].set_visible(False)

        fig.suptitle(f'3-way Venn: Bulk / Pearson / Spearman  –  {cond}  |  p < {PVAL_CUTOFF}',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        fname = OUT_DIR / f'venn3_{cond}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fname.name}')

except Exception as e:
    print(f'venn3 skipped: {e}')

## 9. Correlation scatter: Bulk corr vs NucSeq corr (겹치는 유전자)

In [ ]:
for cond in ['Normal', 'MASLD']:
    bulk_sig  = bulk_normal_sig if cond == 'Normal' else bulk_masld_sig
    bulk_sub  = bulk_df.set_index('symbol')

    for method in METHODS:
        n_ct  = len(CELLTYPES)
        ncols = 4
        nrows = (n_ct + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                  figsize=(ncols * 4, nrows * 4))
        axes = axes.flatten()

        for i, ct in enumerate(CELLTYPES):
            ax        = axes[i]
            ns_df     = nucseq.get((method, ct))
            nucseq_sig = get_nucseq_sig(method, ct, cond)
            overlap   = bulk_sig & nucseq_sig

            if not overlap or ns_df is None:
                ax.set_visible(False)
                continue

            ns_sub    = ns_df.set_index('gene')
            ov_genes  = [g for g in overlap if g in bulk_sub.index and g in ns_sub.index]

            bx = bulk_sub.loc[ov_genes, f'{cond}_spearman_r'].values
            ny = ns_sub.loc[ov_genes,   f'{cond.lower()}_corr'].values

            ax.scatter(bx, ny, alpha=0.6, s=20, color='steelblue', edgecolors='none')
            if len(bx) > 1:
                m, b = np.polyfit(bx, ny, 1)
                xr   = np.linspace(bx.min(), bx.max(), 100)
                ax.plot(xr, m * xr + b, 'r--', linewidth=1)
            ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
            ax.axvline(0, color='gray', linewidth=0.5, linestyle=':')
            ax.set_xlabel('Bulk Spearman r', fontsize=9)
            ax.set_ylabel(f'NucSeq {method} r', fontsize=9)
            ax.set_title(f'{ct}\n(n={len(ov_genes)})', fontsize=10, fontweight='bold')

        for j in range(n_ct, len(axes)):
            axes[j].set_visible(False)

        fig.suptitle(f'Bulk vs NucSeq correlation  [{method} | {cond}]',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        fname = OUT_DIR / f'corr_scatter_{method}_{cond}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fname.name}')

In [ ]:
print('\n=== 분석 완료 ===')
print(f'저장 위치: {OUT_DIR}')
print('\n생성 파일 목록:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')